In [1]:
import sys
import pickle

# TO CHANGE
BASEDIR = "../../"
sys.path.insert(0, BASEDIR)

In [2]:
from src.graph_main import RemoteKnowledgeGraph, RemoteKnowledgeGraphConfig
from src.knowledge_graph_model import GraphModelConfig, EmbeddingsModelConfig
from src.db_drivers.graph_driver import GraphDriverConfig, GraphDBConnectionConfig, DEFAULT_INMEMORYGRAPH_CONFIG
from src.db_drivers.vector_driver import VectorDriverConfig, EmbedderModelConfig, VectorDBConnectionConfig
from src.db_drivers.kv_driver import KeyValueDriverConfig, KVDBConnectionConfig, DEFAULT_INMEMORYKV_CONFIG

from src.qa_pipeline import QAPipelineConfig
from src.qa_pipeline.query_parser import QueryLLMParserConfig
from src.qa_pipeline.knowledge_comparator import KnowledgeComparatorConfig

from src.qa_pipeline.knowledge_retriever import KnowledgeRetrieverConfig
from src.qa_pipeline.knowledge_retriever.AStarTripletsRetriever import AStarGraphSearchConfig
from src.qa_pipeline.knowledge_retriever.BFSTripletsRetriever import BFSSearchConfig

from src.qa_pipeline.answer_generator import QALLMGeneratorConfig

from src.memorize_pipeline import MemPipelineConfig, LLMExtractorConfig, LLMUpdatorConfig

from src.utils import Logger, ReaderMetrics

/home/dzigen/Desktop/PersonalAI/pai_venv/lib/python3.10/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


#### 1. Загружем датасет с триплетами, на основе которого будет построен граф знаний

In [3]:
PKL_GRAPH_PATH = '../../data/pickled_graphs/testdb.pickle'

with open(PKL_GRAPH_PATH, 'rb') as f:
    formated_triplets = pickle.load(f)

print(len(formated_triplets))

10355


#### 2. Задаём конфигурацию графа знаний

In [4]:
inmemory_kg_config = RemoteKnowledgeGraphConfig(
    
    graph_struct_config=GraphModelConfig(driver_config=GraphDriverConfig(
        db_vendor='inmemory_graph', db_config=DEFAULT_INMEMORYGRAPH_CONFIG)), # TO CHANGE
    
    embedds_struct_config=EmbeddingsModelConfig(
        nodesdb_driver_config=VectorDriverConfig(db_vendor='chroma', db_config=VectorDBConnectionConfig(
            path='../../data/graph_structures/vectorized_nodes/testing_testdb', db_name='vectorized_nodes', need_to_clear=False)), # TO CHANGE
        tripletsdb_driver_config=VectorDriverConfig(db_vendor='chroma', db_config=VectorDBConnectionConfig(
            path='../../data/graph_structures/vectorized_triplets/testing_testdb', db_name='vectorized_triplets', need_to_clear=False)), # TO CHANGE
        embedder_config=EmbedderModelConfig(model_name_or_path='../../models/intfloat/multilingual-e5-small')),
    
    qa_pipeline_config=QAPipelineConfig(
        query_parser_config=QueryLLMParserConfig(),
        knowledge_comparator_config=KnowledgeComparatorConfig(),
        knowledge_retriever_config=KnowledgeRetrieverConfig(
            retriever_method='astar', # TO CHANGE
            retriever_config=AStarGraphSearchConfig(), # TO CHANGE
            cache_config=KeyValueDriverConfig(db_vendor='inmemory_kv', db_config=DEFAULT_INMEMORYKV_CONFIG)), # TO CHANGE
        answer_generator_config=QALLMGeneratorConfig(lang='eng')),

    mem_pipeline_config=MemPipelineConfig(
        extractor_config=LLMExtractorConfig(),
        updator_config=LLMUpdatorConfig()),

    log=Logger('log/main'))

#### 3. Инициализируем граф знаний

In [5]:
rkg_main = RemoteKnowledgeGraph(config=inmemory_kg_config)

No sentence-transformers model found with name ../../models/intfloat/multilingual-e5-small. Creating a new one with mean pooling.


In [6]:
# ATTENTION !!!
#rkg_main.kg_model.graph_struct.db_conn.execute_query("match (a) -[r] -> () delete a, r")
#rkg_main.kg_model.graph_struct.db_conn.execute_query("match (a) delete a")
# ATTENTION !!!

[]

#### 4. Добавляем в граф загруженные триплеты

In [6]:
print("uploading data to graph-storage")
rkg_main.kg_model.graph_struct.create_triplets(formated_triplets)
print("uploading data to vector-storage")
rkg_main.kg_model.embeddings_struct.add_triplets(formated_triplets)

uploading data to graph-storage


100%|██████████| 10355/10355 [00:00<00:00, 164716.75it/s]


uploading data to vector-storage


100%|██████████| 81/81 [01:04<00:00,  1.25it/s]


#### 5. Q&A

In [1]:
METRICS = ReaderMetrics(base_dir="../..", bs_model_path="google/electra-base-discriminator")
# METRICS.exact_match(gen_answers, trgt_answers)

In [7]:
qa_examples = [
  ("What is Lauren's opinion about the 13promax battery compared to other batteries in terms of battery life?",
  'Lauren thinks the 13promax battery has better battery life than other batteries in mobile phones.'),
  ("What does Emily think about the performance of her brother's GT2PRO and IQOO7 smartphones during voice calls.",
  'Emily thinks GT2PRO is not as hot during voice calls compared to IQOO7.'),
  ("What is the difference between the heat dissipation performance of the IQOO9 and Xiaomi Mi 12Pro smartphones according to Laura's experience?",
  'Laura finds the heat dissipation performance of the IQOO9 superior to the Xiaomi Mi 12Pro, resulting in less heating while using it for long periods.'),
  ("What are the reasons behind Rodrigo's dislike for Xiaomi's MIUI software, specifically mentioning the frequent bugs he has experienced over the past two years?",
  "Rodrigo dislikes Xiaomi's MIUI software due to frequent bugs he has experienced over the past two years, including small bugs that affect usage, such as force restarts."),
  ("What is Margaret's main concern regarding her MIX4 smartphone?",
  "problem with the photography camera module and its poor performance."),
  ('What are the advantages of the Xiaomi Mi 10pro mentioned by Horace and Jesse, especially in comparison to other devices?',
  'Advantages of Xiaomi Mi 10pro: splitscreen capability, fast charging, screen durability, waterproofing, good signal strength, fast gaming performance. Comparatively better than other devices (especially Apple) in terms of battery life and signal strength.'),
  ('What issues have James and Hailey experienced with their MIX4 devices, especially regarding the photography camera module?',
  "James experienced multiple crashes in a short period, while Hailey mentioned her device being used many times."),
  ("What was Kevin's experience with his brother's GT2PRO and Xiaomi 10Pro?",
  "Kevin found his brother's GT2PRO less hot than expected during voice calls, while Xiaomi 10Pro didn't heat up when recording videos."),
  ("What mobile phone does Kayla's brother have that doesn't get hot during voice calls, according to her?",
  "IQOO7")]

In [8]:
rkg_main.answer_question(qa_examples[0][0])

'Negative'